# 00 · Pilot: Praha

**Cíl:** Ověřit technický pipeline, vizuálně pochopit data, rozhodnout otevřené parametry.

**Otázky k zodpovězení:**
1. Jak vypadá distribuce délek uličních segmentů v Praze jako celku?
2. Liší se centrum (Praha 1) od suburbů (Letňany)?
3. Jaký `network_type` použít: `drive` nebo `all`?
4. Totéž pro plochy městských bloků (H2).

> **Architektura:** Tento notebook je čistý orchestrátor. Veškerá logika je v `src/urbanmal/`.

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import osmnx as ox

from src.urbanmal import viz
from src.urbanmal.data.osm import download_street_network, download_urban_blocks, get_graph_edges
from src.urbanmal.metrics.segments import compute_segment_stats
from src.urbanmal.metrics.blocks import compute_block_stats

viz.set_style()
print("Prostředí OK")

---
## 1 · Praha jako celek – uliční síť

Stáhneme celé administrativní území Prahy, obě varianty `network_type`, a porovnáme.

In [ ]:
PLACE_PRAHA = "Prague, Czech Republic"

G_drive = download_street_network(PLACE_PRAHA, network_type="drive")
G_all   = download_street_network(PLACE_PRAHA, network_type="all")

stats_drive = compute_segment_stats(G_drive)
stats_all   = compute_segment_stats(G_all)

print("network_type='drive':", stats_drive)
print("network_type='all':  ", stats_all)

In [ ]:
edges_drive = get_graph_edges(G_drive)
edges_all   = get_graph_edges(G_all)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
viz.plot_segment_distribution(edges_drive["length"], "Praha (drive)", ax=axes[0])
viz.plot_segment_distribution(edges_all["length"],   "Praha (all)",   ax=axes[1])
plt.tight_layout()
plt.savefig("../reports/figures/01_praha_segment_dist_drive_vs_all.png", dpi=150)
plt.show()

---
## 2 · Centrum vs. suburb – uliční segmenty

Srovnáme tři oblasti:
- **Praha 1** – historické centrum
- **Letňany** – panelákový suburb
- **Zličín** – okrajová zástavba / nákupní zóna

In [ ]:
AREAS = {
    "Praha 1": "Praha 1, Czech Republic",
    "Letňany": "Letňany, Prague, Czech Republic",
    "Zličín":  "Zličín, Prague, Czech Republic",
}

graphs  = {name: download_street_network(place, network_type="drive") for name, place in AREAS.items()}
stats   = {name: compute_segment_stats(G) for name, G in graphs.items()}
edges   = {name: get_graph_edges(G) for name, G in graphs.items()}

for name, s in stats.items():
    print(f"{name:12s}  průměr={s['mean_m']:.1f} m  medián={s['median_m']:.1f} m  n={s['count']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, lengths) in zip(axes, {k: v["length"] for k, v in edges.items()}.items()):
    viz.plot_segment_distribution(lengths, name, ax=ax)
plt.tight_layout()
plt.savefig("../reports/figures/02_centrum_vs_suburb_segments.png", dpi=150)
plt.show()

---
## 3 · Vizualizace sítě na mapě

Rychlý pohled na to, jak sítě vypadají prostorově.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, G) in zip(axes, graphs.items()):
    ox.plot_graph(G, ax=ax, show=False, close=False,
                  node_size=0, edge_linewidth=0.4, edge_color="#444")
    ax.set_title(name)
plt.tight_layout()
plt.savefig("../reports/figures/03_site_mapy.png", dpi=150)
plt.show()

---
## 4 · Městské bloky (H2) – Praha 1 vs. Letňany

Stejné srovnání, ale pro plochy bloků.

In [ ]:
blocks = {name: download_urban_blocks(place) for name, place in list(AREAS.items())[:2]}
block_stats = {name: compute_block_stats(gdf) for name, gdf in blocks.items()}

for name, s in block_stats.items():
    print(f"{name:12s}  průměr={s['mean_m2']:.0f} m²  medián={s['median_m2']:.0f} m²  n={s['count']}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, gdf) in zip(axes, blocks.items()):
    gdf_proj = gdf.to_crs("EPSG:3857")
    areas = gdf_proj.geometry.area.dropna()
    # omezíme na rozumný rozsah (odfiltrujeme extrémní outliers >99. percentil)
    cap = areas.quantile(0.99)
    viz.plot_segment_distribution(areas[areas <= cap], f"{name} – bloky", ax=ax)
    ax.set_xlabel("Plocha bloku (m²)")
plt.tight_layout()
plt.savefig("../reports/figures/04_bloky_centrum_vs_suburb.png", dpi=150)
plt.show()

---
## 5 · Závěry pilotu a rozhodnutí

Po spuštění buněk výše doplň pozorování:

| Rozhodnutí | Volba | Zdůvodnění |
|---|---|---|
| `network_type` | `drive` / `all` | *(doplnit po prohlédnutí grafů)* |
| Agregace segmentů | mean / median | *(doplnit – symetrie distribuce?)* |
| Hranice města | admin. boundary | Konzistentní, OSM ji má pro všechna ČR města |
| Min. počet segmentů | ? | *(doplnit – jaký je min. n pro smysluplný průměr?)* |

**Předběžný závěr H1:**  
*(doplnit: je centrum kratší než suburb? Jak výrazně?)*

**Předběžný závěr H2:**  
*(doplnit: jsou bloky v centru menší? Jak výrazně?)*